# Retrieval-Augmented Generation (RAG)

In [1]:
from setup import bedrock_tool
import chromadb
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the ../rag_setup/rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

2026-03-02 13:49:16.546465723 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [4]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [5]:
# print(calorie_lookup_tool('bananas'))

In [6]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(calorie_lookup_tool.__dict__)],
)

In [7]:
with trace("Nutrition Assistant with RAG") as t:
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 

The banana has 89.0 calories per 100g and the apple has 52.0 calories per 100g. 

To get the total calories, I would need the weights of the banana and apple. However, I can provide the calories per 100g for both fruits.

Banana: 89.0 calories per 100g
Apple: 52.0 calories per 100g

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_08a3baa19a5246f5a26209fe97b158a1


## Health-Related Nutritional Q&A

Now let's create a system to answer health-related nutritional questions using the `nutrition_qna` collection.

In [8]:
# Connect to the nutrition Q&A collection
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")
print(f"Nutrition Q&A database has {nutrition_qna.count()} Q&A pairs")

Nutrition Q&A database has 5197 Q&A pairs


In [9]:
# Test query to see what kind of answers we get
test_results = nutrition_qna.query(query_texts=["vitamin D deficiency"], n_results=2)
for i, doc in enumerate(test_results["documents"][0]):
    print(f"Result {i+1}:")
    print(doc)
    print("\n")

Result 1:
Question: What could be the reason for low levels of vitamin D?
        Answer: Vitamin D deficiency can occur due to various factors such as limited exposure to sunlight, unhealthy dietary habits leading to insufficient calcium and Vitamin D intake, pollution affecting skin's ability to synthesize vitamin D, use of phytates and phosphates which deplete vitamin D stores, increased melanin in the skin reducing vitamin D production, and cultural practices like wearing burqas that limit sun exposure.

        This Q&A pair provides information about nutrition and health topics.


Result 2:
Question: What are some health consequences of vitamin D deficiency?
        Answer: One may suffer from defective calcification of bones, muscle weakness, boned based pain as a result of insufficient levels of vitamin D. This nutrient plays an important role in maintaining proper balance between calcium and phosphorus.

        This Q&A pair provides information about nutrition and health top

In [10]:
@function_tool
def nutrition_qna_tool(question: str, max_results: int = 3) -> str:
    """
    Tool to answer health-related nutritional questions using a Q&A database.
    
    Args:
        question: The health or nutrition question to answer.
        max_results: The maximum number of similar Q&A pairs to return.
    
    Returns:
        A string containing relevant answers from the nutrition Q&A database.
    """
    
    results = nutrition_qna.query(query_texts=[question], n_results=max_results)
    
    if not results["documents"][0]:
        return f"No information found for: {question}"
    
    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(f"Answer {i+1}: {doc}")
    
    return "Relevant Information:\n" + "\n\n".join(formatted_results)

In [11]:
health_nutrition_agent = Agent(
    name="Health Nutrition Expert",
    instructions="""
    You are a knowledgeable health and nutrition expert.
    You provide accurate, evidence-based answers to health-related nutritional questions.
    When asked a question, use the nutrition_qna_tool to search for relevant information.
    Synthesize the information from multiple sources if available and provide a clear, concise answer.
    Always be careful to give accurate health information.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(nutrition_qna_tool.__dict__)],
)

### Test the Health Nutrition Agent

Let's ask some health-related nutritional questions:

In [12]:
# Example 1: Ask about vitamin D deficiency
with trace("Health Question - Vitamin D") as t:
    result = await Runner.run(
        health_nutrition_agent,
        "What are the symptoms of vitamin D deficiency?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 

Vitamin D deficiency can manifest in several ways, with symptoms often being vague and nonspecific. Common symptoms include:

1. Bone Pain: Chronic bone pain and muscle aches are often the first signs of vitamin D deficiency, particularly in weight-bearing joints like the hips and legs.

2. Muscle Weakness: Vitamin D is crucial for muscle function. A deficiency can lead to muscle weakness and poor muscle tone.

3. Fatigue: Feeling unusually tired or fatigued can be a sign of vitamin D deficiency. This is often due to the role of vitamin D in energy production.

4. Mood Changes: Low levels of vitamin D have been associated with mood disorders such as depression and anxiety.

5. Poor Immune Function: Vitamin D plays a role in immune function. A deficiency can increase susceptibility to infections.

6. Bone Health Issues: Long-term deficiency can lead to bone diseases such as rickets in children and osteomalacia or osteoporosis in adults.

7. Hair Loss: Some people with vitamin 

In [13]:
# Example 2: Ask about pregnancy nutrition
with trace("Health Question - Pregnancy") as t:
    result = await Runner.run(
        health_nutrition_agent,
        "What nutritional advice is important during pregnancy?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 

Here is the nutritional advice that is important during pregnancy:

1. **Iron**: Essential for the production of hemoglobin, which carries oxygen in the blood. Iron deficiency can lead to anemia, which is common in pregnancy.

2. **Calcium**: Important for the development of the baby's bones and teeth. It also helps to maintain the mother's bone density.

3. **Protein**: Necessary for the growth of fetal tissue, including the brain, and for the increase in blood volume that occurs during pregnancy.

4. **Vitamin A**: Important for the development of the baby's eyes and various body tissues.

5. **Iodine**: Essential for the production of thyroid hormones, which are important for the baby's brain development.

6. **Energy**: Pregnant women need to consume more calories overall to support the growth of the baby and the changes in the mother's body.

Ensuring a balanced diet that includes these nutrients is crucial for a healthy pregnancy. It's also important to consult with a h

In [14]:
# Example 3: Ask about dietary fiber
with trace("Health Question - Fiber") as t:
    result = await Runner.run(
        health_nutrition_agent,
        "What are the health benefits of dietary fiber?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 

The health benefits of dietary fiber are multifaceted:

1. **Chronic Disease Prevention**: A high-fiber diet may lower the risk of chronic degenerative diseases such as colon cancer and cardiovascular disease. It may also reduce serum estrogen levels, potentially decreasing the likelihood of hormone-related cancers like endometrial, ovarian, and prostate cancers.

2. **Weight Management**: Fiber-rich foods tend to be more filling, which can help reduce overall calorie intake and support weight loss efforts. 

3. **Digestive Health**: Fiber adds bulk to meals, making bowel movements softer and easier, and can help prevent constipation. It also slows the absorption of nutrients, which can be beneficial for people with diabetes by helping to maintain stable blood sugar levels.

4. **Overall Wellness**: Fiber is essential for proper bowel function and is commonly found in a variety of whole foods such as fruits, vegetables, whole grains, and legumes.

Including a variety of fiber